# Notebook 3B: Zero-Shot Cross-Target Transfer

Train on one GPCR, predict on another -- no retraining, no fine-tuning, no
domain adaptation. Explicitly NOT a core research question for this paper
(notebooks 2-5 already answer those); this is a bounded, secondary analysis
enabled cheaply because notebook 3 preserved consistent global feature
definitions, scaffold IDs, and frozen model artifacts.

**Scope:**
- Full pool only (not ki/ki_ic50).
- One pre-selected algorithm (XGBoost, combined representation -- the
  nested-CV mean-performance winner across most target/pool combinations
  per notebook 3's findings, `10/15` classification and `12/15` regression).
- Frozen source models reused directly (loaded from `ml/models/`), no new
  per-pair Optuna search.
- 20 directed source->destination pairs (5 targets, all ordered pairs
  excluding self-pairs) x 2 tasks (classification + regression) = 40
  evaluations total.

**Essential controls, computed for every evaluation, not just reported in
aggregate:**
1. Exact-compound-overlap control -- evaluate with and without destination
   compounds that are exact-SMILES matches to something in the source's own
   training set.
2. Scaffold-overlap control -- evaluate separately for destination compounds
   whose Bemis-Murcko scaffold was/was not present in the source's training
   scaffolds (`global_scaffold_id`, already computed and consistent across
   targets from notebook 2).
3. Source-domain applicability-domain status per destination compound --
   using the SOURCE model's own frozen AD reference (scaler + kNN), not a
   destination-specific one.
4. Shared-compound label-consistency check -- for exact-overlap compounds,
   whether the source and destination target actually agree on active/
   inactive for that compound (genuine cross-target selectivity is real
   biology, not a transfer failure, and must not be silently folded into
   the headline transfer accuracy).
5. Destination baseline comparison -- zero-shot transfer performance framed
   against the destination target's own properly-trained model performance
   (`final_model_test_performance.csv`, same canonical test split), not
   left to stand alone.

Framing is explicitly **zero-shot**, not few-shot or domain-adaptation --
the destination target's compounds are never touched during training,
calibration, or conformal-wrapper fitting, all of which stay frozen from
the source target's own notebook-3 run.

In [ ]:
# MUST BE FIRST CELL!
import os
import multiprocessing

HPC_MODE = True

if HPC_MODE:
    N_CORES = int(os.environ.get("NCPUS") or os.environ.get("PBS_NP") or
                  os.environ.get("PBS_NCPUS") or os.environ.get("SLURM_CPUS_PER_TASK") or
                  multiprocessing.cpu_count())
else:
    N_CORES = min(multiprocessing.cpu_count(), 4)

# Keep math libraries single-threaded on HPC -- same convention as notebook 3.
# This job has no joblib/sklearn CV parallelism to hand N_CORES to (pure
# model inference, not training), so this mainly guards against XGBoost's
# own internal thread pool oversubscribing the allocation.
for var in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS",
            "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[var] = "1" if HPC_MODE else str(N_CORES)

os.environ["OMP_NESTED"] = "FALSE"
os.environ["MKL_DYNAMIC"] = "FALSE"

ENV = "HPC" if HPC_MODE else "Colab"
print(f"Environment: {ENV} | N_CORES: {N_CORES}")


In [ ]:
from pathlib import Path

if HPC_MODE:
    PROJECT_DIR = Path("./")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/My Drive/gpcr_benchmark")

for subdir in ["ml/results", "ml/models", "data/processed"]:
    (PROJECT_DIR / subdir).mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")


In [ ]:
# rdkit/sklearn/xgboost/joblib already present in bioenv -- no install step
# needed here, same as notebook 3 (its own pip-install line is commented out
# for the identical reason).
import json
import hashlib
import datetime
import platform
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import venn_abers  # required to unpickle notebook 3's VennAbers calibrator objects

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

from sklearn.metrics import roc_auc_score, r2_score

print('Imports OK.')
print('Python:', platform.python_version())
for pkg in ('rdkit', 'numpy', 'pandas', 'sklearn', 'xgboost', 'joblib', 'venn_abers'):
    try:
        mod = __import__(pkg)
        print(f'{pkg}: {getattr(mod, "__version__", "unknown")}')
    except ImportError:
        print(f'{pkg}: NOT INSTALLED')

models_dir  = PROJECT_DIR / 'ml' / 'models'
data_dir    = PROJECT_DIR / 'data' / 'processed'
results_dir = PROJECT_DIR / 'ml' / 'results'
out_dir     = PROJECT_DIR / 'ml' / 'results' / 'zeroshot_transfer'
out_dir.mkdir(parents=True, exist_ok=True)

TARGETS = ['drd2', 'cb2', 'adora2a', 'oprm1', 'ccr5']
ALGORITHM = 'xgb'
REPRESENTATION = 'combined'
POOL = 'full'
CLASS_THRESHOLD = 0.5

def log_milestone(message: str):
    log_path = out_dir / 'run_log_03b_zeroshot.txt'
    with open(log_path, 'a') as f:
        f.write(f'[{datetime.datetime.now().isoformat()}] {message}\n')
    return log_path

print(f'\nTargets: {TARGETS}')
print(f'Fixed algorithm: {ALGORITHM}, representation: {REPRESENTATION}, pool: {POOL}')


## Module A: Load Frozen Source Model Artifacts

One set of artifacts per target, all from `{target}_full/combined/`:
classifier + regressor (frozen, trained on that target's own full-pool
data), their calibrator/conformal wrappers, the AD reference (scaler +
fitted kNN), and the feature-name schema (confirmed identical across all
5 targets -- one universal feature-computation function works for every
source/destination pair, only the model objects differ per source).

In [ ]:
def load_source_artifacts(target: str) -> dict:
    '''Loads every frozen artifact notebook 3 produced for one target's
    full-pool, combined-representation model. Raises loudly (not silently
    skipped) if any expected file is missing -- a partial artifact set for
    a target used as a SOURCE would silently corrupt every pair it appears
    in as source.'''
    base = models_dir / f'{target}_full' / 'combined'
    required = [
        'xgb_clf.joblib', 'xgb_reg.joblib',
        'xgb_clf_calibrator.pkl', 'xgb_clf_conformal.pkl', 'xgb_reg_conformal.pkl',
        'ad_reference.pkl', 'feature_names.json',
    ]
    missing = [f for f in required if not (base / f).exists()]
    if missing:
        raise FileNotFoundError(f'{target}: missing artifact(s) {missing} in {base}')

    return {
        'clf': joblib.load(base / 'xgb_clf.joblib'),
        'reg': joblib.load(base / 'xgb_reg.joblib'),
        'clf_calibrator': joblib.load(base / 'xgb_clf_calibrator.pkl'),
        'clf_conformal': joblib.load(base / 'xgb_clf_conformal.pkl'),
        'reg_conformal': joblib.load(base / 'xgb_reg_conformal.pkl'),
        'ad_reference': joblib.load(base / 'ad_reference.pkl'),
        'feature_names': json.load(open(base / 'feature_names.json')),
    }


source_artifacts = {}
for t in TARGETS:
    source_artifacts[t] = load_source_artifacts(t)
    print(f'  loaded {t}: clf n_features_in_={getattr(source_artifacts[t]["clf"], "n_features_in_", "?")}')

# Confirmed identical across all 5 targets -- one shared feature schema.
FEATURE_NAMES = source_artifacts[TARGETS[0]]['feature_names']
for t in TARGETS[1:]:
    assert source_artifacts[t]['feature_names'] == FEATURE_NAMES, (
        f'{t} has a different feature schema than {TARGETS[0]} -- '
        f'universal feature computation assumption is violated, stop and investigate.'
    )
print(f'\nFeature schema confirmed identical across all targets: {len(FEATURE_NAMES)} features '
      f'({FEATURE_NAMES[:3]} ... {FEATURE_NAMES[-2:]})')


## Module B: Load Destination Compound Datasets

`cleaned_data_{target}_full.csv` (notebook 2 output) for all 5 targets --
these serve as BOTH the destination compound pool (every target, when
playing destination) AND the source-side compound-overlap reference (every
target, when playing source). `compound_scaffold_assignments.csv` filtered
to `activity_pool=='full'` supplies the global, cross-target-consistent
scaffold IDs needed for the scaffold-overlap control.

In [ ]:
cleaned_data = {}
for t in TARGETS:
    path = data_dir / f'cleaned_data_{t}_full.csv'
    assert path.exists(), f'{path} not found -- run notebook 2 first.'
    df = pd.read_csv(path)
    assert df['clean_smiles'].duplicated().sum() == 0, f'{t}: duplicate clean_smiles found, expected none'
    cleaned_data[t] = df
    print(f'{t}: {len(df)} compounds, activity balance {df["activity"].mean():.3f}')

scaffold_assignments_path = data_dir / 'compound_scaffold_assignments.csv'
assert scaffold_assignments_path.exists(), f'{scaffold_assignments_path} not found -- run notebook 2 first.'
scaffold_all = pd.read_csv(scaffold_assignments_path)
scaffold_full = scaffold_all[scaffold_all['activity_pool'] == POOL].copy()

# Per-target lookup: clean_smiles -> global_scaffold_id, and the target's
# own full set of scaffold IDs (source side of the scaffold-overlap control).
scaffold_lookup = {}
scaffold_sets = {}
for t in TARGETS:
    sub = scaffold_full[scaffold_full['target'] == t]
    scaffold_lookup[t] = dict(zip(sub['clean_smiles'], sub['global_scaffold_id']))
    scaffold_sets[t] = set(sub['global_scaffold_id'])
    print(f'{t}: {len(scaffold_sets[t])} unique scaffolds ({POOL} pool)')


## Module C: Feature Computation (Universal Schema)

10 physicochemical descriptors (already present as columns in
`cleaned_data_*_full.csv`, computed once by notebook 2) + 2048-bit Morgan
fingerprint (radius 2, computed here at runtime from `clean_smiles`) =
2058 features, in the exact order every frozen model expects -- confirmed
identical across targets in Module A, so this function is reused unchanged
for every source/destination pair.

In [ ]:
DESCRIPTOR_COLS = FEATURE_NAMES[:10]
MORGAN_RADIUS = 2
MORGAN_NBITS = len(FEATURE_NAMES) - len(DESCRIPTOR_COLS)
assert MORGAN_NBITS == 2048, f'Expected 2048 Morgan bits, got {MORGAN_NBITS}'


def compute_morgan_bits(smiles_list):
    '''Radius-2, 2048-bit Morgan fingerprint per SMILES. Returns an
    (n_compounds, 2048) array; a row of zeros for any SMILES RDKit fails
    to parse (fails loud via the returned parse-failure mask, not silently
    dropped -- caller decides how to handle).'''
    n = len(smiles_list)
    bits = np.zeros((n, MORGAN_NBITS), dtype=np.int8)
    parsed_ok = np.zeros(n, dtype=bool)
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=MORGAN_RADIUS, nBits=MORGAN_NBITS)
        arr = np.zeros(MORGAN_NBITS, dtype=np.int8)
        Chem.DataStructs = __import__('rdkit.DataStructs', fromlist=['ConvertToNumpyArray'])
        Chem.DataStructs.ConvertToNumpyArray(fp, arr)
        bits[i] = arr
        parsed_ok[i] = True
    return bits, parsed_ok


def build_feature_matrix(df: pd.DataFrame) -> tuple:
    '''df must have the 10 descriptor columns + clean_smiles. Returns
    (X, parsed_ok) with X in FEATURE_NAMES order -- rows failing SMILES
    parsing are flagged in parsed_ok, not silently zero-filled and kept.'''
    descriptor_X = df[DESCRIPTOR_COLS].to_numpy(dtype=np.float64)
    morgan_X, parsed_ok = compute_morgan_bits(df['clean_smiles'].tolist())
    X = np.hstack([descriptor_X, morgan_X])
    return X, parsed_ok


print('Feature computation function defined:', len(DESCRIPTOR_COLS), 'descriptors +', MORGAN_NBITS, 'Morgan bits')


## Module D: Directed Pairs, Essential Controls, Zero-Shot Prediction

20 directed (source, destination) pairs -- every ordered pair of the 5
targets excluding self-pairs. For each pair: compute the destination
compounds' features once, tag every essential control (compound overlap,
scaffold overlap, label agreement where shared, source-AD status), then
predict with the source's frozen classifier + regressor. No retraining,
no fine-tuning -- destination compounds are used only at inference time.

Checkpointed per pair so a Colab disconnect does not lose completed pairs,
same convention as every other notebook in this project.

In [ ]:
def compute_source_ad_status(X_scaled_source_space: np.ndarray, ad_reference: dict) -> np.ndarray:
    '''kNN-distance applicability domain, using the SOURCE model's own
    frozen scaler + fitted NearestNeighbors (never refit on destination
    data). mean distance to k nearest SOURCE-training neighbors, compared
    against the source's own stored threshold.'''
    knn = ad_reference['knn']
    k = ad_reference['k']
    threshold = ad_reference['threshold']
    distances, _ = knn.kneighbors(X_scaled_source_space, n_neighbors=k)
    mean_dist = distances.mean(axis=1)
    return mean_dist <= threshold, mean_dist


DIRECTED_PAIRS = [(s, d) for s in TARGETS for d in TARGETS if s != d]
assert len(DIRECTED_PAIRS) == 20, f'Expected 20 directed pairs, got {len(DIRECTED_PAIRS)}'
print(f'{len(DIRECTED_PAIRS)} directed source->destination pairs')

predictions_path = out_dir / 'zeroshot_predictions.csv'
if predictions_path.exists():
    predictions_records = pd.read_csv(predictions_path).to_dict('records')
    _done_pairs = {(r['source'], r['destination']) for r in predictions_records}
    print(f'Checkpoint found: {len(_done_pairs)} pair(s) already done, loading from disk')
else:
    predictions_records = []
    _done_pairs = set()

for source, destination in DIRECTED_PAIRS:
    if (source, destination) in _done_pairs:
        print(f'  \u23ed\ufe0f  {source} -> {destination}: checkpoint found, skipping')
        continue

    dest_df = cleaned_data[destination].copy()
    X_dest, parsed_ok = build_feature_matrix(dest_df)
    dest_df = dest_df[parsed_ok].reset_index(drop=True)
    X_dest = X_dest[parsed_ok]
    if (~parsed_ok).sum() > 0:
        log_milestone(f'{source}->{destination}: {(~parsed_ok).sum()} destination compounds '
                       f'failed SMILES parsing, excluded')

    art = source_artifacts[source]

    # ---- essential control 1: exact-compound overlap (source-side) ----
    source_smiles_set = set(cleaned_data[source]['clean_smiles'])
    is_shared_compound = dest_df['clean_smiles'].isin(source_smiles_set).to_numpy()

    # ---- essential control 4: shared-compound label agreement ----
    source_label_lookup = dict(zip(cleaned_data[source]['clean_smiles'], cleaned_data[source]['activity']))
    dest_labels = dest_df['activity'].to_numpy()
    label_agrees = np.full(len(dest_df), np.nan)
    for i, smi in enumerate(dest_df['clean_smiles']):
        if smi in source_label_lookup:
            label_agrees[i] = float(source_label_lookup[smi] == dest_labels[i])

    # ---- essential control 2: scaffold overlap (source-side) ----
    source_scaffold_set = scaffold_sets[source]
    dest_scaffold_ids = dest_df['clean_smiles'].map(scaffold_lookup[destination])
    is_seen_scaffold = dest_scaffold_ids.isin(source_scaffold_set).to_numpy()

    # ---- essential control 3: source-domain AD status ----
    X_dest_scaled = art['ad_reference']['scaler'].transform(X_dest)
    within_source_ad, source_ad_mean_dist = compute_source_ad_status(X_dest_scaled, art['ad_reference'])

    # ---- zero-shot prediction, source model, destination compounds ----
    raw_proba_2col = art['clf'].predict_proba(X_dest)
    raw_proba = raw_proba_2col[:, 1]
    # Calibrator is now a VennAbers object (notebook 3 swapped Platt ->
    # Venn-ABERS) -- takes the full 2-column predict_proba output directly,
    # returns (p_prime, p0_p1). p0_p1 width is a real per-compound
    # calibration-uncertainty signal, persisted below rather than discarded.
    p_prime, p0_p1 = art['clf_calibrator'].predict_proba(raw_proba_2col)
    calibrated_proba = p_prime[:, 1]
    calibration_interval_width = p0_p1[:, 1] - p0_p1[:, 0]
    predicted_class = (calibrated_proba >= CLASS_THRESHOLD).astype(int)
    predicted_pactivity = art['reg'].predict(X_dest)

    for i in range(len(dest_df)):
        predictions_records.append({
            'source': source, 'destination': destination,
            'clean_smiles': dest_df['clean_smiles'].iloc[i],
            'molecule_chembl_id': dest_df['molecule_chembl_id'].iloc[i],
            'true_activity': int(dest_labels[i]),
            'true_pactivity': float(dest_df['pActivity'].iloc[i]),
            'predicted_class': int(predicted_class[i]),
            'calibrated_proba': float(calibrated_proba[i]),
            'calibration_interval_width': float(calibration_interval_width[i]),
            'predicted_pactivity': float(predicted_pactivity[i]),
            'is_source_shared_compound': bool(is_shared_compound[i]),
            'shared_compound_label_agrees': label_agrees[i] if not np.isnan(label_agrees[i]) else None,
            'is_source_seen_scaffold': bool(is_seen_scaffold[i]),
            'within_source_ad': bool(within_source_ad[i]),
            'source_ad_mean_dist': float(source_ad_mean_dist[i]),
        })

    pd.DataFrame(predictions_records).to_csv(predictions_path, index=False)  # checkpoint after each pair
    n_shared = int(is_shared_compound.sum())
    n_seen_scaf = int(is_seen_scaffold.sum())
    n_in_ad = int(within_source_ad.sum())
    print(f'  \u2705 {source} -> {destination}: {len(dest_df)} compounds -- '
          f'{n_shared} source-shared, {n_seen_scaf} seen-scaffold, {n_in_ad} within source AD')

predictions_df = pd.DataFrame(predictions_records)
print(f'\nTotal predictions: {len(predictions_df)} rows across {predictions_df.groupby(["source","destination"]).ngroups} pairs')


## Module E: Stratified Transfer-Performance Metrics

Every directed pair scored 4 ways, not one headline number: **all**
destination compounds, **excluding** source-shared compounds (the
compound-overlap control), **unseen-scaffold only** (the scaffold-overlap
control), and **within-source-AD only**. A pair that only "transfers well"
because it is scored mostly on compounds the source has already
memorized is a different (and much less interesting) result than one that
transfers well on genuinely unseen chemistry.

In [ ]:
def safe_roc_auc(y_true, y_score):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def safe_r2(y_true, y_pred):
    if len(y_true) < 2:
        return np.nan
    return r2_score(y_true, y_pred)


strata = {
    'all': lambda df: pd.Series(True, index=df.index),
    'excl_source_shared': lambda df: ~df['is_source_shared_compound'],
    'unseen_scaffold_only': lambda df: ~df['is_source_seen_scaffold'],
    'within_source_ad_only': lambda df: df['within_source_ad'],
}

transfer_metrics_records = []
for (source, destination), grp in predictions_df.groupby(['source', 'destination']):
    for stratum_name, stratum_fn in strata.items():
        mask = stratum_fn(grp)
        sub = grp[mask]
        transfer_metrics_records.append({
            'source': source, 'destination': destination, 'stratum': stratum_name,
            'n_compounds': len(sub),
            'roc_auc': safe_roc_auc(sub['true_activity'], sub['calibrated_proba']),
            'r2': safe_r2(sub['true_pactivity'], sub['predicted_pactivity']),
        })

transfer_metrics_df = pd.DataFrame(transfer_metrics_records)
transfer_metrics_path = out_dir / 'zeroshot_transfer_metrics.csv'
transfer_metrics_df.to_csv(transfer_metrics_path, index=False)

print(transfer_metrics_df[transfer_metrics_df['stratum'] == 'all']
      [['source', 'destination', 'n_compounds', 'roc_auc', 'r2']].to_string(index=False))
print(f'\nSaved: {transfer_metrics_path} ({len(transfer_metrics_df)} rows -- '
      f'20 pairs x 4 strata = {20 * 4} expected)')


## Module F: Destination Baseline Comparison

Zero-shot transfer performance means little standing alone -- framed here
against the destination target's own properly-trained model, from
`final_model_test_performance.csv` (notebook 3's single canonical
train/test split, same evaluation philosophy as this notebook's own
single-pass zero-shot evaluation, so a fair like-for-like comparison
rather than mixing in nested-CV's lower-variance estimate).

In [ ]:
baseline_path = results_dir / 'final_model_test_performance.csv'
assert baseline_path.exists(), f'{baseline_path} not found -- run notebook 3 first.'
baseline_df = pd.read_csv(baseline_path)

# final_model_test_performance.csv stores algorithm as a display name
# ('XGBoost', 'Random Forest', 'LightGBM'), not the short code ('xgb') used
# for model-directory paths elsewhere in this notebook -- confirmed live
# after this exact mismatch caused a 0-row filter on first run. ALGORITHM
# itself stays 'xgb' everywhere else (correct for the models/ paths).
ALGORITHM_DISPLAY_NAMES = {'xgb': 'XGBoost', 'rf': 'Random Forest', 'lgb': 'LightGBM'}

destination_baseline = baseline_df[
    (baseline_df['activity_pool'] == POOL)
    & (baseline_df['feature_representation'] == REPRESENTATION)
    & (baseline_df['algorithm'] == ALGORITHM_DISPLAY_NAMES[ALGORITHM])
][['target', 'test_roc_auc', 'test_r2']].rename(
    columns={'target': 'destination', 'test_roc_auc': 'destination_baseline_roc_auc',
             'test_r2': 'destination_baseline_r2'}
)
print(destination_baseline.to_string(index=False))
assert len(destination_baseline) == len(TARGETS), (
    f'Expected {len(TARGETS)} destination baselines, got {len(destination_baseline)} -- '
    f'check algorithm/representation/pool naming matches final_model_test_performance.csv exactly.'
)


## Module G: Final Consolidated Results + Manifest

Joins the 'all' stratum of the transfer metrics against the destination
baseline for the headline transfer-vs-baseline comparison table, keeps the
full stratified table (all 4 strata) as the complete, disclosed result --
main text can cite the headline join, SI/reviewers get the full
compound-overlap / scaffold-overlap / AD-status breakdown per pair.

In [ ]:
headline_df = transfer_metrics_df[transfer_metrics_df['stratum'] == 'all'].merge(
    destination_baseline, on='destination', how='left'
)
headline_df['roc_auc_gap_vs_destination_baseline'] = (
    headline_df['roc_auc'] - headline_df['destination_baseline_roc_auc']
)
headline_df['r2_gap_vs_destination_baseline'] = (
    headline_df['r2'] - headline_df['destination_baseline_r2']
)

headline_path = out_dir / 'zeroshot_transfer_headline.csv'
headline_df.to_csv(headline_path, index=False)

print(headline_df[['source', 'destination', 'n_compounds', 'roc_auc',
                    'destination_baseline_roc_auc', 'roc_auc_gap_vs_destination_baseline',
                    'r2', 'destination_baseline_r2', 'r2_gap_vs_destination_baseline']]
      .sort_values('roc_auc_gap_vs_destination_baseline', ascending=False)
      .to_string(index=False))

mean_gap_auc = headline_df['roc_auc_gap_vs_destination_baseline'].mean()
mean_gap_r2 = headline_df['r2_gap_vs_destination_baseline'].mean()
print(f'\nMean gap vs destination baseline across all 20 pairs: '
      f'ROC-AUC {mean_gap_auc:+.3f}, R2 {mean_gap_r2:+.3f}')
print(f'Saved: {headline_path}')


In [ ]:
# =============================================================================
# MANIFEST -- SHA-256-hashed, same convention as notebooks 2-5.
# =============================================================================
def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def _git_commit():
    try:
        import subprocess
        return subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=str(PROJECT_DIR),
                                        stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        return None

manifest_outputs = {}
for name, path in [
    ('zeroshot_predictions.csv', predictions_path),
    ('zeroshot_transfer_metrics.csv', transfer_metrics_path),
    ('zeroshot_transfer_headline.csv', headline_path),
]:
    if path.exists():
        manifest_outputs[name] = {'path': str(path), 'sha256': _sha256(path), 'n_rows': sum(1 for _ in open(path)) - 1}

manifest = {
    'timestamp': datetime.datetime.now().isoformat(),
    'git_commit': _git_commit(),
    'python_version': platform.python_version(),
    'config': {
        'targets': TARGETS, 'algorithm': ALGORITHM, 'representation': REPRESENTATION,
        'pool': POOL, 'class_threshold': CLASS_THRESHOLD,
        'n_directed_pairs': len(DIRECTED_PAIRS), 'strata': list(strata.keys()),
        'scope_note': ('Zero-shot only -- no retraining, no fine-tuning, no destination data '
                        'used at training/calibration/conformal-fitting time. Frozen source '
                        'models reused directly from ml/models/.'),
    },
    'outputs': manifest_outputs,
    'package_versions': {},
}
for pkg in ('rdkit', 'numpy', 'pandas', 'sklearn', 'xgboost', 'joblib'):
    try:
        manifest['package_versions'][pkg] = getattr(__import__(pkg), '__version__', 'unknown')
    except ImportError:
        pass

manifest_path = out_dir / 'manifest_03b_zeroshot_transfer.json'
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, default=str)
print(f'Manifest saved: {manifest_path}')
print('\nNotebook 3B (zero-shot cross-target transfer) complete.')
